# 00 · Preprocessing minimal
Input: `data/raw/bi-usd-rate.csv`, `data/raw/cnn_internasional.csv` (tidak diubah).
Output di `data/processed/`: `articles.parquet`, `daily.parquet`, `splits.json`, `meta.json`.

In [1]:
import re, json, hashlib
import numpy as np, pandas as pd
from utils import RAW, PROC, fingerprint
PROC.mkdir(parents=True, exist_ok=True)

END_DATE = None       # mis. '2026-11-30' kalau kurs sudah diperpanjang; None = semua kurs yang ada
CUTOFF_HOUR = 15      # berita >= 15:00 WIB dipakai hari bursa berikutnya
MIN_WORDS = 30        # ambang artikel terlalu pendek (ditetapkan sekali)

## Kurs
`kurs_mid` = rata-rata Kurs Jual dan Kurs Beli.

In [2]:
k = pd.read_csv(RAW / 'bi-usd-rate.csv')
k['date'] = pd.to_datetime(k['Tanggal'], format='%m/%d/%Y %I:%M:%S %p').dt.normalize()
k = k.sort_values('date').drop_duplicates('date')
if END_DATE: k = k[k.date <= END_DATE]
k['kurs_mid'] = (k['Kurs Jual'] + k['Kurs Beli']) / 2
daily = k[['date', 'kurs_mid']].reset_index(drop=True)
daily['ret'] = np.log(daily.kurs_mid).diff()
daily['target'] = daily.ret.shift(-1)
print(daily.date.min().date(), '->', daily.date.max().date(), len(daily), 'hari kurs')

2021-09-01 -> 2026-09-01 1202 hari kurs


## Artikel: tanggal, pembersihan, filter
Tanggal kosong dipulihkan dari 14 digit di URL (`yyyymmddHHMMSS`), diasumsikan WIB. Cek dulu selisihnya dengan `date` yang ada.

In [3]:
a = pd.read_csv(RAW / 'cnn_internasional.csv').rename(columns={'date': 'published_at', 'content': 'full_text'})
a = a[['title', 'published_at', 'url', 'full_text']]
a['published_at'] = pd.to_datetime(a.published_at, utc=True).dt.tz_convert('Asia/Jakarta')
url_ts = pd.to_datetime(a.url.str.extract(r'(\d{14})')[0], format='%Y%m%d%H%M%S').dt.tz_localize('Asia/Jakarta')

gap_min = (a.published_at - url_ts).dt.total_seconds() / 60
print('published_at - waktu URL (menit):\n', gap_min.describe().round(1).to_string())
assert gap_min.median() < 180, 'selisih median terlalu besar, zona waktu URL kemungkinan salah'
print('tanggal kosong dipulihkan dari URL:', a.published_at.isna().sum())
a['published_at'] = a.published_at.fillna(url_ts)

published_at - waktu URL (menit):
 count       8819.0
mean        1650.3
std        31922.7
min            1.3
25%           18.3
50%           45.5
75%          153.2
max      1857701.8
tanggal kosong dipulihkan dari URL: 1171


In [4]:
# Pola boilerplate dari pengamatan teks (baris yang berulang), bukan dari label. Dibekukan.
BOILER = [r'\[Gambas:[^\]]*\]', r'(?mi)^\s*Bersambung ke halaman berikutnya\.*\s*$', r'(?mi)^\s*Lanjut ke sebelah\.*\s*$',
          r'(?mi)^\s*Baca di halaman berikutnya\s*>*\s*$', r'(?mi)^\s*Baca selengkapnya di sini\s*\.?\s*$',
          r'^\s*Jakarta, CNN Indonesia\s*--\s*']

def clean(t):
    for p in BOILER: t = re.sub(p, '', t)
    t = re.sub(r'<[^>]+>', ' ', t)
    return re.sub(r'[\s\x00-\x1f\x7f-\x9f]+', ' ', t).strip()   # huruf besar/kecil dan tanda baca tidak diubah

a['text'] = a.full_text.map(clean)
n0 = len(a)
a = a.sort_values('published_at')
a = a[~a.url.duplicated() & ~a.title.str.strip().str.lower().duplicated()]
a = a[a.text.str.split().str.len() >= MIN_WORDS].reset_index(drop=True)
a.insert(0, 'article_id', np.arange(len(a)))
print(f'{n0} -> {len(a)} artikel setelah dedup + filter panjang')

9990 -> 9224 artikel setelah dedup + filter panjang


## Penyelarasan ke hari kurs dan split
Berita < 15:00 WIB → hari itu; ≥ 15:00 → hari berikutnya. Akhir pekan/libur → hari kurs berikutnya (`searchsorted`).

In [5]:
tdi = pd.DatetimeIndex(daily.date)
d0 = a.published_at.dt.tz_localize(None).dt.normalize() + pd.to_timedelta((a.published_at.dt.hour >= CUTOFF_HOUR).astype(int), unit='D')
idx = tdi.searchsorted(d0)
a['aligned_trading_date'] = pd.Series(tdi[np.minimum(idx, len(tdi) - 1)], index=a.index).where(idx < len(tdi))

cut = (a.aligned_trading_date + pd.Timedelta(hours=CUTOFF_HOUR)).dt.tz_localize('Asia/Jakarta')
assert (a.aligned_trading_date.isna() | (a.published_at <= cut)).all(), 'ada berita yang masuk hari terlalu awal'

# Jendela: dari hari kurs pertama yang punya berita sampai hari terakhir yang punya target. Kronologis 70/15/15.
win = daily[(daily.date >= a.aligned_trading_date.min()) & daily.target.notna()].date.reset_index(drop=True)
i1, i2 = int(len(win) * .70), int(len(win) * .85)
daily['split'] = 'none'
for name, days in (('train', win[:i1]), ('val', win[i1:i2]), ('test', win[i2:])):
    daily.loc[daily.date.isin(days), 'split'] = name

splits = {s: dict(start=str(g.date.min().date()), end=str(g.date.max().date()), n=len(g))
          for s, g in daily[daily.split != 'none'].groupby('split')}
splits['hash'] = hashlib.sha256(','.join(win.dt.strftime('%Y-%m-%d')).encode()).hexdigest()
f = PROC / 'splits.json'
if f.exists():
    assert json.load(open(f))['hash'] == splits['hash'], 'split berubah! splits.json sudah dibekukan, hapus manual kalau memang disengaja'
else:
    json.dump(splits, open(f, 'w'), indent=1)
print(json.dumps(splits, indent=1))

{
 "test": {
  "start": "2026-07-29",
  "end": "2026-08-31",
  "n": 22
 },
 "train": {
  "start": "2026-01-22",
  "end": "2026-06-29",
  "n": 98
 },
 "val": {
  "start": "2026-06-30",
  "end": "2026-07-28",
  "n": 21
 },
 "hash": "ad6de88800c6c6b6ee9b77e28dbcb0b5767085aa887e01d5b9563729fb7407eb"
}


In [6]:
a.to_parquet(PROC / 'articles.parquet')
daily.to_parquet(PROC / 'daily.parquet')
json.dump(dict(n=len(a), fingerprint=fingerprint(a)), open(PROC / 'meta.json', 'w'))
print('artikel tidak masuk split mana pun:', (~a.aligned_trading_date.isin(win)).sum())
print('hari di jendela tanpa berita:', (~win.isin(a.aligned_trading_date)).sum(), 'dari', len(win))

artikel tidak masuk split mana pun: 2661
hari di jendela tanpa berita: 0 dari 141
